### ECM Models

This notebook covers all currently available ECM models and their behavior. The exact implementation can be seen in `src.models.submodels.ecm`. We begin by importing the necessary libraries and modules.

In [2]:
import sys
import os
# setup paths
sys.path.append(os.path.abspath(".."))

from src.models.submodels.ecm import GalerkinPCE, nRC

Currently, two types of models are supported in the ECM class: nRC which is the standard implementation of an n-th order RC ECM, and GalerkinPCE which uses Polynomial Chaos to approximate a 2nd order ECM with gaussian parameters. Here's how you instantiate both: 

In [3]:
ecm_nrc = nRC(N=2)
ecm_pce = GalerkinPCE()

[nRC] Initialized!
[GalerkinPCE] Initialized!


Just like OCV models, ECM models require parametrization data to function. If instantiated without a `config` argument, they fall back to the default data in the [`../data`](../data) directory.

If the user wants to load their own parametrization data, they can provide a configuration dictionary with the following structure:

```yaml
paths:
    parameters:
        dist_folder: "path/to/distribution/parquet"
        file_naming_scheme: "{temp}deg.parquet" 
        # interpolators replace {temp} with temperatures and load files
        param_labels: 
        - 'mu_R0'
        - 'sigma_R0'
        - 'mu_tau1_inv'
        - 'sigma_tau1_inv'
        - 'mu_c1_inv'
        - 'sigma_c1_inv'
        - 'mu_tau2_inv'
        - 'sigma_tau2_inv'
        - 'mu_c2_inv'
        - 'sigma_c2_inv'  # parameter names to read from distribution files

To model the SoC- and temperature-dependent behavior, each ECM model instantiates a `BatteryParameterInterpolator` object (implementation in [`src/gaussian_process/interpolators.py`](/src/gaussian_process/interpolators.py)), which uses the data to provide interpolated parameter values during integration.

- `dist_folder` is the directory containing the parametrization data. It is generally expected to contain a dataframe with SoC (0–100) as the index and the parameters at those SoC thresholds as columns, as illustrated below:

![Distribution File Example](../assets/distribution_file.png)

- `file_naming_scheme` tells the interpolator how to read the files.  
- `param_labels` should match the column names in your distribution files and dictate the keys of the dictionary returned by the interpolator at each query.